# Danish Agriculture Agency Playground

This notebook is meant as a playground to get familiar with the [Danish Agriculture Agency](https://landbrugsgeodata.fvm.dk/) different datasets.

In [ ]:
import ee
import geemap
import os
from dotenv import load_dotenv
import numpy as np
load_dotenv()

# Initialize the Earth Engine module.
PROJECT_ID = os.getenv("PROJECT_ID")
ee.Initialize(project = os.getenv("PROJECT_ID"))

Earth Engine only cares about .shp, .shx, .dbf and .prj files. Important to clarify that our dataset follows `ISO 88591` and not the standard `UTF-8`. To use the dataset as a an Earth Engine Object we first need to upload it to [EE Code Editor](https://code.earthengine.google.com/). `geemap.shp_to_ee`  could have also been an otpion but the CRS had to be EPSG:4326. 


In [ ]:
dataset = ee.FeatureCollection("projects/" + PROJECT_ID + "/assets/Marker24")

display(dataset.limit(5))

## Exploring the DAA Dataset

In [ ]:
crop_categories = dataset.aggregate_array('AfgKat').distinct().getInfo()
print(f"Crop categories", crop_categories)
print(f"Number of crop categories", len(crop_categories))

crop_names = dataset.aggregate_array('AfgNavn').distinct().getInfo()
print(f"Crop Names", crop_names)
print(f"Number of crop names", len(crop_names))

crop_codes = dataset.aggregate_array('AfgNr').distinct().getInfo()
# print(f"Crop codes", crop_codes)
print(f"Number of crop codes", len(crop_codes))

owners = dataset.aggregate_array('EjerNr').distinct().getInfo()
# print(f"Owners", owners)
print(f"Number of owners", len(owners))

field_blocks = dataset.aggregate_array('MarkBlok').distinct().getInfo()
# print(f"Field Blocks", field_blocks)
print(f"Number of field blocks", len(field_blocks))

field_numbers = dataset.aggregate_array('MarkNr').distinct().getInfo()
# print(f"Field Numbers", field_numbers)
print(f"Number of field numbers", len(field_numbers))

print(f"Number of features: 617271")
# print(f"Number of features:", dataset.size().getInfo())


In [ ]:
field_blocks_all = dataset.aggregate_array('MarkBlok').length()
display(field_blocks_all)

In [ ]:
def add_area(feature):
    return feature.set('area_m2', feature.geometry().area())

with_area = dataset.map(add_area)

total_area = with_area.aggregate_sum('area_m2').getInfo()

In [ ]:
field_area_km = total_area / pow(10,6)
denmark_area_km = 43094

print(f"Total Field Area in Denmark (km²):", field_area_km)
print(f"Total Area of Denmark (km²):", denmark_area_km)
print(f"Percentage of Denmark which is fields:", field_area_km/denmark_area_km * 100)

## Merging with Dynamic World

In [ ]:
START = ee.Date('2024-06-01')
END = START.advance(1, 'month')
# END = START.advance(1, 'year')
DENMARK = ee.FeatureCollection('USDOS/LSIB_SIMPLE/2017').filter(ee.Filter.eq('country_na', 'Denmark')).geometry()
GEOMETRY = dataset.geometry()

# define filter
col_filter = ee.Filter.And(
    ee.Filter.bounds(DENMARK), 
    ee.Filter.date(START, END),
)

# Get relevant DW images
dw_col = ee.ImageCollection('GOOGLE/DYNAMICWORLD/V1').filter(col_filter)

# Calculate Yearly mode
# most_frequent_class = dw_col.select('label').reduce(ee.Reducer.mode())
# most_frequent_class = most_frequent_class.rename(['label'])


In [ ]:
# Export Task
# task = ee.batch.Export.image.toAsset(
#     image=most_frequent_class,
#     description='DW_Yearly_Mode_2024',
#     assetId='projects/' + PROJECT_ID + '/assets/dw_2024_mode', 
#     scale=10,
#     region= DENMARK,
#     maxPixels=1e13
# )

# task.start()

In [ ]:
cloud_filter = ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 35)

s2_col = ee.ImageCollection('COPERNICUS/S2_HARMONIZED').filter(cloud_filter)
linked_col = dw_col.linkCollection(s2_col, s2_col.first().bandNames())

In [ ]:
# Create map
map = geemap.Map(scroll_wheel_zoom = False)

DW_CLASS = {
    'water' : '419bdf', # blue
    'trees' : '397d49', # green
    'grass' : '88b053', # light green
    'flooded_vegetation' : '7a87c6', # greyish blue
    'crops' : 'e49635', # orange
    'shrub_and_scrub' : 'dfc35a', # yellow
    'built' : 'c4281b', # red
    'bare' : 'a59b8f', # grey
    'snow_and_ice' : 'b39fe1', # purple
}

DW_VIZ_PARAMS = {
  'min' : 0,
  'max' : 8,
  'palette' : list(DW_CLASS.values()),
  'bands' : 'label'
}

# B2, B3, B4 are the optical visualization bands, 
# min defines the values that should be mapped to RGB(0) and max the values that should be mapped to RGB(255)
S2_VIZ_PARAMS = {
  'min' : 0,
  'max' : 3000,
  'bands' : ['B4', 'B3', 'B2']
}



In [ ]:
saved_image = ee.Image('projects/' + PROJECT_ID + '/assets/dw_2024_mode')

In [ ]:
img = linked_col.mosaic()
# map.centerObject(img, 12)

# Add Sentinel-2 layer 
map.add_layer(
    img,
    S2_VIZ_PARAMS, 
    'Sentinel-2 L1C',
)
# Add DW classification layer
map.add_layer(
    saved_image,
    DW_VIZ_PARAMS,
    'Dynamic World',
)
map.add_layer(dataset, {'color': 'white', 'oppacity': 0.2}, "Polygon Boundaries")
map